In [1]:
PREFIJO_SAVE = "resultados_iniciales"
import os

SEED = 42


def set_seed(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "1"

exper_config = {
    "num_rounds": 100,
    "num_clients": 10,
    "random_seed": 42,
    "batch_size": 32,

    "lr_cliente_simulado": 0.001,

    "num_shadow_models": 50,  # Tiene que ser par
    "global_model_epochs": 10,
    "shadow_train_rounds": 10,
    "shadow_data_fraction": 0.01,
    "lr_shadow_training": 0.001,
    "batch_size_attack_model": 8,

    "prob_range": 0.1,
    "fraction": 0.4,

    "aleatoriedad": False,

    # ATAQUE #
    "epochs_ataque": 50,
    "lr_ataque": 0.0001,

    # RUIDO #
    "aplicar_ruido": False,
    "ruido_obj": ["gradients"],
    "ruido_per": 0.4,  # Proporción de datos afectados por ruido
    "noise_std": 0.2,  # Desviación estándar del ruido
    "epsilon": 1.0,  # Parámetro de privacidad diferencial
    "delta": 1e-5,  # Delta para ruido gaussiano
    "sensitivity": 1.0,  # Sensibilidad del mecanismo de ruido
    "privacy_type": "gaussian",  # Tipo de ruido a aplicar
    "selected_layers": "all",  # [0, -1]  # Aplica ruido a todas las capas

    # LABEL FLIPPING #
    "label_flipping": False,
    "flipping_antes": False,
    "prob_flip_0": 0.2,
    "prob_flip_1": 0.2,
    "flip_target": "Slice",

    # RESULTADOS #
    "rondas_a_ignorar": 10,
    "property_threshold": 0.5,
    "learning_rate": 0.1,
    "data_file_path": "/home/iagobg/notebooks/label_bi_10.csv"  # Ruta al archivo
    ,
    "prefijo_save": "resultados_iniciales",
}


In [2]:
import os
import logging

#### LOG ####
LOG_DIR = 'logs'
if not os.path.exists(LOG_DIR):
    os.makedirs(LOG_DIR)

if not os.path.exists(exper_config["prefijo_save"]):
    os.makedirs(exper_config["prefijo_save"])

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(LOG_DIR, 'execution.log')),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)

In [3]:


set_seed()


def load_data(file_path):
    """
    Carga un archivo CSV y devuelve un DataFrame.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"El archivo {file_path} no se encontró.")

    df = pd.read_csv(file_path)
    return df


def preprocess_data(df, fraction=exper_config["fraction"]):
    """
    Preprocesa los datos, filtra columnas necesarias y aplica muestreo.
    """
    necessary_columns = [
        'Src IP', 'Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet',
        'Fwd Packet Length Std', 'ACK Flag Count', 'Fwd Seg Size Min', 'label', 'Slice'
    ]

    df = df[necessary_columns].dropna()

    if exper_config["aleatoriedad"]:
        df = df.sample(frac=fraction).reset_index(drop=True)
    else:
        df = df.sample(frac=fraction, random_state=SEED).reset_index(drop=True)

    X = df.drop(['label', 'Slice'], axis=1).values  # Convertir a numpy
    y_label = df['label'].values  # Convertir a numpy
    y_slice = df['Slice'].values  # Convertir a numpy

    return X, y_label, y_slice


def create_client_data(X, y_label, y_slice):
    """
    Divide los datos para los clientes y aplica ruido o flipping si está configurado.
    """
    num_clients = exper_config["num_clients"]
    logger.info(f"Creating data for {num_clients} clients")

    X_with_property = X[y_slice == 1]
    y_label_with_property = y_label[y_slice == 1]
    X_without_property = X[y_slice == 0]
    y_label_without_property = y_label[y_slice == 0]

    if num_clients % 2 != 0:
        raise ValueError("El número de clientes debe ser par para balancear propiedades.")

    min_data_size = min(len(X_with_property) // (num_clients // 2), len(X_without_property) // (num_clients // 2))

    client_data = []
    for i in range(num_clients // 2):
        client_data.append({
            'X': torch.tensor(X_with_property[i * min_data_size:(i + 1) * min_data_size], dtype=torch.float32),
            'y_label': torch.tensor(y_label_with_property[i * min_data_size:(i + 1) * min_data_size],
                                    dtype=torch.float32),
            'y_slice': 1,
            'has_property': True
        })
        client_data.append({
            'X': torch.tensor(X_without_property[i * min_data_size:(i + 1) * min_data_size], dtype=torch.float32),
            'y_label': torch.tensor(y_label_without_property[i * min_data_size:(i + 1) * min_data_size],
                                    dtype=torch.float32),
            'y_slice': 0,
            'has_property': False
        })

    return client_data


def split_data_for_models(X, y_label, y_slice):
    """
    Divide los datos en conjuntos para el modelo global y los modelos sombra.
    """
    logger.info("Dividiendo datos para el modelo global y los modelos sombra.")

    num_shadow_samples = int(len(X) * exper_config['shadow_data_fraction'])
    indices = np.arange(len(X))

    if exper_config["aleatoriedad"]:
        np.random.shuffle(indices)

    shadow_indices = indices[:num_shadow_samples]
    global_indices = indices[num_shadow_samples:]

    X_shadow, y_label_shadow, y_slice_shadow = X[shadow_indices], y_label[shadow_indices], y_slice[shadow_indices]
    X_global, y_label_global, y_slice_global = X[global_indices], y_label[global_indices], y_slice[global_indices]

    logger.info(f"Datos divididos: {len(X_global)} para el modelo global, {len(X_shadow)} para los modelos sombra.")

    return (X_global, y_label_global, y_slice_global), (X_shadow, y_label_shadow, y_slice_shadow), len(X_shadow)


In [4]:



def select_clients(round_num, clients_with_property, clients_without_property):
    """
    Selecciona los clientes de forma determinista:
    - En rondas pares: Se eligen clientes con la propiedad.
    - En rondas impares: Se eligen clientes sin la propiedad.
    """
    return clients_with_property if round_num % 2 == 0 else clients_without_property


def initialize_clients(client_data, global_model, global_model_epochs, batch_size):
    def init_client(client_id, data):
        # Pasar el modelo global completo
        return SimulatedFlowerClient(client_id, data, global_model, global_model_epochs, batch_size)

    # Paralelizar la inicialización de los clientes
    if exper_config["aleatoriedad"]:
        with ThreadPoolExecutor(max_workers=10) as executor:
            futures = [executor.submit(init_client, i, data) for i, data in enumerate(client_data)]
            clients = [f.result() for f in futures]
    else:
        clients = [init_client(i, data) for i, data in enumerate(client_data)]

    return clients


class SimulatedFlowerClient:
    def __init__(self, client_id, data, model, config, batch_size):
        """
        Inicializa un cliente simulado con sus datos y modelo en Federated Learning.
        """
        self.client_id = client_id
        self.data = data
        self.X = data["X"].clone().detach() if isinstance(data["X"], torch.Tensor) else torch.tensor(data["X"].values,
                                                                                                     dtype=torch.float32) if isinstance(
            data["X"], pd.DataFrame) else torch.tensor(data["X"], dtype=torch.float32)
        self.y_label = data["y_label"].clone().detach() if isinstance(data["y_label"], torch.Tensor) else torch.tensor(
            data["y_label"], dtype=torch.float32)
        self.config = config
        self.batch_size = batch_size

        # Inicializar el modelo copiando el global
        self.model = model
        self.model.load_state_dict(model.state_dict())
        self.optimizer = optim.Adam(self.model.parameters(), lr=exper_config["lr_cliente_simulado"])
        self.loss_fn = nn.BCELoss()

    def get_parameters(self):
        """Devuelve los parámetros actuales del modelo del cliente."""
        return self.model.state_dict()

    def fit(self, global_weights):
        """
        Entrena el modelo localmente y devuelve las actualizaciones.
        """
        self.model.load_state_dict(global_weights)
        self.model.train()

        for epoch in range(self.config["global_model_epochs"]):
            self.optimizer.zero_grad()
            outputs = self.model(self.X)
            loss = self.loss_fn(outputs.squeeze(), self.y_label)
            loss.backward()
            self.optimizer.step()

        new_weights = self.model.state_dict()
        updates = {}
        for key in new_weights.keys():
            gw = global_weights[key]
            # si es np array, conviene cast a torch
            if not isinstance(gw, torch.Tensor):
                gw = torch.tensor(gw, dtype=torch.float32)

            updates[key] = new_weights[key] - gw.clone().detach()

        return new_weights, len(self.X), {"updates": updates}

    def evaluate(self, global_weights):
        """
        Evalúa el modelo global en los datos locales del cliente.
        """
        self.model.load_state_dict(global_weights)
        self.model.eval()
        with torch.no_grad():
            X_input = self.X if isinstance(self.X, torch.Tensor) else torch.tensor(self.X, dtype=torch.float32)
            outputs = self.model(X_input)

            loss = self.loss_fn(outputs.squeeze(), self.y_label)
            accuracy = ((outputs.squeeze() > 0.5) == self.y_label).float().mean().item()

        return loss.item(), len(self.X), {"accuracy": accuracy}


In [5]:

import torch.nn as nn


def create_global_model(input_shape):
    """
    Crea un modelo de red neuronal para aprendizaje federado en PyTorch.

    Parameters:
        input_shape (int): Número de características de entrada.

    Returns:
        torch.nn.Module: Modelo de PyTorch.
    """

    class GlobalModel(nn.Module):
        def __init__(self, input_shape):
            super(GlobalModel, self).__init__()
            self.fc1 = nn.Linear(input_shape, 128)
            self.bn1 = nn.BatchNorm1d(128)
            self.fc2 = nn.Linear(128, 64)
            self.fc3 = nn.Linear(64, 1)
            self.dropout = nn.Dropout(0.3)

        def forward(self, x):
            x = F.relu(self.bn1(self.fc1(x)))
            x = self.dropout(x)
            x = F.relu(self.fc2(x))
            x = self.dropout(x)
            x = torch.sigmoid(self.fc3(x))
            return x

    logger.info(f"Creando modelo global con input shape {input_shape}")
    return GlobalModel(input_shape)


def create_shadow_model(input_shape):
    """
    Crea un modelo sombra para inferencia de propiedades en PyTorch.

    Parameters:
        input_shape (int): Dimensión de entrada.

    Returns:
        torch.nn.Module: Modelo de PyTorch.
    """

    class ShadowModel(nn.Module):
        def __init__(self, input_shape):
            super(ShadowModel, self).__init__()
            self.fc1 = nn.Linear(input_shape, 64)
            self.fc2 = nn.Linear(64, 32)
            self.fc3 = nn.Linear(32, 1)
            self.dropout = nn.Dropout(0.3)

        def forward(self, x):
            x = F.relu(self.fc1(x))
            x = self.dropout(x)
            x = F.relu(self.fc2(x))
            x = self.dropout(x)
            x = torch.sigmoid(self.fc3(x))
            return x

    logger.info(f"Creando modelo sombra con input shape {input_shape}")
    return ShadowModel(input_shape)


class AttackModelNN(nn.Module):
    def __init__(self, input_dim):
        super(AttackModelNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 1)
        # Si deseas más capas, agrégalas.

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))  # Sale prob en [0,1]
        return x

    def train_model(self, X_train, y_train, epochs, lr, batch_size):
        """
        Entrena el modelo AttackModelNN en un dataset (X_train, y_train) con PyTorch.
        :param X_train: np.array con shape [N, input_dim]
        :param y_train: np.array con shape [N]
        :param epochs: número de épocas
        :param lr: learning rate
        :param batch_size: batch size
        """
        # 1) Convertir a tensores
        X_tensor = torch.tensor(X_train, dtype=torch.float32)
        y_tensor = torch.tensor(y_train, dtype=torch.float32)

        dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

        optimizer = torch.optim.Adam(self.parameters(), lr=lr)
        loss_fn = nn.BCELoss()

        self.train()  # modo train
        for ep in range(epochs):
            total_loss = 0.0
            for batch_x, batch_y in dataloader:
                optimizer.zero_grad()
                out = self.forward(batch_x).squeeze(dim=1)  # [BS]
                loss = loss_fn(out, batch_y)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * len(batch_x)
            avg_loss = total_loss / len(dataset)
            print(f"Epoch {ep + 1}/{epochs}, Loss={avg_loss:.6f}")

    def predict_proba(self, X_input_np):
        """
        Dado un np.array shape (N, input_dim),
        retorna probabilities shape (N, 2) al estilo scikit-learn
        donde la 2a columna es la prob. de clase=1
        """
        self.eval()
        X_t = torch.tensor(X_input_np, dtype=torch.float32)
        with torch.no_grad():
            outputs = self.forward(X_t).squeeze(dim=1)  # shape (N,)
            # outputs es prob de la clase=1
            prob_1 = outputs.detach().cpu().numpy()
        prob_0 = 1.0 - prob_1
        # unimos [prob_0, prob_1] en shape (N,2)
        probs = np.stack([prob_0, prob_1], axis=1)
        return probs


def train_shadow_models_sin_epoch(X_shadow, y_label_shadow, y_slice_shadow, global_model):
    """
    Entrena varios modelos sombra, separando los datos con propiedad (Slice=1)
    y sin propiedad (Slice=0). Además, recopila actualizaciones etiquetadas
    para entrenar luego un Attack Model.

    Return:
      shadow_models (list): la lista de shadow models entrenados.
      X_attack (list of np arrays): lista de vectores de actualización etiquetados
      y_attack (list of int): etiquetas 0/1 indicando si el vector de actualización
                              corresponde a 'prop=0' o 'prop=1'
    """
    logger.info(f"Training {exper_config['num_shadow_models']} shadow models prior to federated learning.")

    shadow_models = []
    X_attack = []
    y_attack = []

    # Separamos los datos de shadow en 2 subconjuntos
    X_prop = X_shadow[y_slice_shadow == 1]
    y_prop = y_label_shadow[y_slice_shadow == 1]

    X_noprop = X_shadow[y_slice_shadow == 0]
    y_noprop = y_label_shadow[y_slice_shadow == 0]

    # Suponemos que exper_config['num_shadow_models'] es par,
    # la mitad entrenan en "prop" y la mitad en "noprop":
    num_sm = exper_config['num_shadow_models']
    half = num_sm // 2

    loss_fn = nn.BCELoss()

    # Entrenamos half shadow models con "prop=1"
    for i in range(half):
        logger.info(f"Training shadow model (PROP) {i + 1}/{half}")

        shadow_model = type(global_model)(global_model.fc1.in_features)
        shadow_model.load_state_dict(global_model.state_dict())
        shadow_model.train()

        optimizer = optim.Adam(shadow_model.parameters(), lr=exper_config["lr_shadow_training"])

        # Entrenamiento sencillo
        for epoch in range(exper_config['shadow_train_rounds']):
            optimizer.zero_grad()
            out = shadow_model(X_prop)
            loss = loss_fn(out.squeeze(), y_prop)
            loss.backward()
            optimizer.step()

        shadow_models.append(shadow_model)

        # Guardar vector de actualización final
        final_sd = shadow_model.state_dict()
        update_vec_list = []
        for k, v in final_sd.items():
            update_vec_list.append(v.flatten())
        merged_vec = torch.cat(update_vec_list, dim=0)

        X_attack.append(merged_vec.detach().cpu().numpy())
        y_attack.append(1)  # 1 => propiedad presente

    # Entrenamos half shadow models con "prop=0"
    for i in range(half):
        logger.info(f"Training shadow model (NOPROP) {i + 1}/{half}")

        shadow_model = type(global_model)(global_model.fc1.in_features)
        shadow_model.load_state_dict(global_model.state_dict())
        shadow_model.train()

        optimizer = optim.Adam(shadow_model.parameters(), lr=exper_config["lr_shadow_training"])

        for epoch in range(exper_config['shadow_train_rounds']):
            optimizer.zero_grad()
            out = shadow_model(X_noprop)
            loss = loss_fn(out.squeeze(), y_noprop)
            loss.backward()
            optimizer.step()

        shadow_models.append(shadow_model)

        # Guardar vector de actualización final
        final_sd = shadow_model.state_dict()
        update_vec_list = []
        for k, v in final_sd.items():
            update_vec_list.append(v.flatten())
        merged_vec = torch.cat(update_vec_list, dim=0)

        X_attack.append(merged_vec.detach().cpu().numpy())
        y_attack.append(0)  # 0 => sin propiedad

    logger.info("All shadow models trained successfully. Building X_attack, y_attack dataset.")
    return shadow_models, X_attack, y_attack


def train_shadow_models(
        X_shadow,
        y_label_shadow,
        y_slice_shadow,
        global_model,
        shadow_train_rounds=10
):
    """
    Entrena varios modelos sombra, separando los datos con propiedad (Slice=1)
    y sin propiedad (Slice=0). Además, en cada epoch se guarda el vector de
    actualización, para tener más datos al entrenar el modelo de ataque.

    Return:
      shadow_models (list): Lista de shadow models entrenados.
      X_attack (list): Lista de vectores de actualización etiquetados (in/out).
      y_attack (list): Etiquetas 0/1 indicando si el vector pertenece a 'prop=1' o 'prop=0'.
    """
    import torch
    import torch.nn as nn
    import torch.optim as optim

    logger.info(f"Training {exper_config['num_shadow_models']} shadow models with epoch-level updates.")

    shadow_models = []
    X_attack = []
    y_attack = []

    # Separamos datos en prop=1 y prop=0
    X_prop = X_shadow[y_slice_shadow == 1]
    y_prop = y_label_shadow[y_slice_shadow == 1]

    X_noprop = X_shadow[y_slice_shadow == 0]
    y_noprop = y_label_shadow[y_slice_shadow == 0]

    num_sm = exper_config['num_shadow_models']
    half = num_sm // 2  # mitad para 'prop=1', mitad para 'prop=0'

    loss_fn = nn.BCELoss()

    # -------------------------------------
    # 1) Entrenamos 'half' shadow models con 'prop=1'
    # -------------------------------------
    for i in range(half):
        logger.info(f"Training shadow model (PROP) {i + 1}/{half}")

        # Creamos un nuevo modelo sombra con la misma estructura que el global
        shadow_model = type(global_model)(global_model.fc1.in_features)
        shadow_model.load_state_dict(global_model.state_dict())
        shadow_model.train()

        optimizer = optim.Adam(shadow_model.parameters(), lr=0.001)

        # Para calcular actualizaciones por epoch
        prev_sd = {}
        for k, v in shadow_model.state_dict().items():
            prev_sd[k] = v.clone().detach()

        for epoch in range(shadow_train_rounds):
            optimizer.zero_grad()
            out = shadow_model(X_prop)
            loss = loss_fn(out.squeeze(), y_prop)
            loss.backward()
            optimizer.step()

            # Guardar vector de actualización de esta epoch
            current_sd = shadow_model.state_dict()
            # vector de diferencia epoch_i = current - prev
            update_vec_list = []
            for k, v in current_sd.items():
                diff = v - prev_sd[k]
                update_vec_list.append(diff.flatten())
            merged_vec = torch.cat(update_vec_list, dim=0)

            X_attack.append(merged_vec.detach().cpu().numpy())
            # Este update es 'prop=1'
            y_attack.append(1)

            # Actualizar prev_sd para la siguiente epoch
            for k, v in current_sd.items():
                prev_sd[k] = v.clone().detach()

        shadow_models.append(shadow_model)

    # -------------------------------------
    # 2) Entrenamos 'half' shadow models con 'prop=0'
    # -------------------------------------
    for i in range(half):
        logger.info(f"Training shadow model (NOPROP) {i + 1}/{half}")

        shadow_model = type(global_model)(global_model.fc1.in_features)
        shadow_model.load_state_dict(global_model.state_dict())
        shadow_model.train()

        optimizer = optim.Adam(shadow_model.parameters(), lr=0.001)

        # Para calcular actualizaciones por epoch
        prev_sd = {}
        for k, v in shadow_model.state_dict().items():
            prev_sd[k] = v.clone().detach()

        for epoch in range(shadow_train_rounds):
            optimizer.zero_grad()
            out = shadow_model(X_noprop)
            loss = loss_fn(out.squeeze(), y_noprop)
            loss.backward()
            optimizer.step()

            # Guardar vector de actualización de esta epoch
            current_sd = shadow_model.state_dict()
            update_vec_list = []
            for k, v in current_sd.items():
                diff = v - prev_sd[k]
                update_vec_list.append(diff.flatten())
            merged_vec = torch.cat(update_vec_list, dim=0)

            X_attack.append(merged_vec.detach().cpu().numpy())
            # Este update es 'prop=0'
            y_attack.append(0)

            for k, v in current_sd.items():
                prev_sd[k] = v.clone().detach()

        shadow_models.append(shadow_model)

    logger.info("All shadow models trained. Built X_attack, y_attack with epoch-level updates.")
    return shadow_models, X_attack, y_attack


In [6]:

import torch


def aggregate_updates(client_weights, client_sample_counts):
    """
    Realiza una agregación ponderada de los pesos del modelo en función
    del número de muestras de cada cliente.
    """
    total_samples = sum(client_sample_counts)
    aggregated_weights = {}
    for key in client_weights[0].keys():
        # sumamos w_i * (samples_i / total_samples)
        aggregated_weights[key] = sum(
            client_weights[i][key] * (client_sample_counts[i] / total_samples)
            for i in range(len(client_weights))
        )
    return aggregated_weights


class FederatedServer:
    def __init__(self, global_model, clients, attack_model, num_rounds):
        """
        :param global_model: el modelo global (PyTorch) que se entrena federadamente
        :param clients: lista de clientes
        :param attack_model: el modelo de ataque ya entrenado (p.ej. RandomForest),
                             con un método predict_proba() o predict()
        :param num_rounds: número de rondas
        """
        self.global_model = global_model
        self.clients = clients
        self.attack_model = attack_model  # <--- SE RECIBE AQUI
        self.num_rounds = num_rounds

    def train(self):
        """
        Ejecuta el proceso de aprendizaje federado en varias rondas.
        """
        logger.info(f"Starting Federated Learning with {self.num_rounds} rounds")
        results = []

        for round_num in range(1, self.num_rounds + 1):
            logger.info(f"Round {round_num} - Selecting clients for training")
            selected_clients, has_property = self.select_clients(round_num)
            logger.info(f"{len(selected_clients)} clients selected for training. Has Property: {has_property}")

            global_weights = self.global_model.state_dict()

            # 1) Recolectar actualizaciones de los clientes
            client_weights = []
            client_sample_counts = []

            for client in selected_clients:
                updated_weights, sample_count, update_info = client.fit(global_weights)
                client_weights.append(updated_weights)
                client_sample_counts.append(sample_count)

            # 2) Agregamos (average) las actualizaciones y actualizamos el global
            averaged_weights = aggregate_updates(client_weights, client_sample_counts)
            self.global_model.load_state_dict(averaged_weights)

            # 3) Construimos el vector de actualización para cada cliente
            client_updates_vectors = []
            for cw in client_weights:
                # Convertir todas las capas del cliente en un único vector
                param_list = []
                for k, v in cw.items():
                    param_list.append(v.view(-1))  # aplanar
                merged = torch.cat(param_list, dim=0)  # concatenar
                client_updates_vectors.append(merged)

            # 4) Inferencia de la propiedad usando *attack_model*
            #    (ya entrenado con las actualizaciones de los shadow models)
            property_prob = self.infer_property_with_attack(client_updates_vectors)

            # 5) Evaluación del modelo global
            result = self.evaluate(round_num)
            result.update({
                'property_probability': property_prob,
                'has_property': has_property,
                'prediction': property_prob > 0.5,
                'threshold_used': 0.5
            })
            results.append(result)

            logger.info(f"Round {round_num} - Property Probability: {property_prob}")

        return results

    def infer_property_with_attack(self, client_updates_vectors):
        """
        Aplica el modelo de ataque (AttackModel) a los vectores de actualización
        para estimar la probabilidad de que la propiedad esté presente.
        """
        if self.attack_model is None:
            logger.warning("No attack_model provided. Returning 0.5 by default.")
            return 0.5

        # Convertimos cada update a numpy y pedimos prob al attack model
        property_preds = []
        for upd in client_updates_vectors:
            # shape [vector_dim]
            upd_np = upd.cpu().numpy().reshape(1, -1)  # [1, vector_dim]
            # suponemos que attack_model tiene predict_proba
            probs = self.attack_model.predict_proba(upd_np)[0]
            prob_class_1 = probs[1]  # asumiendo 2 clases -> (clase_0, clase_1)
            property_preds.append(prob_class_1)

        # un solo valor: la media
        final_prob = float(np.mean(property_preds))
        return final_prob

    def evaluate(self, round_num):
        y_true = []
        y_pred = []
        all_losses = []

        for client in self.clients:
            loss, num_samples, metrics = client.evaluate(self.global_model.state_dict())
            all_losses.append(loss)

            # extiende y_true
            y_true.extend(client.y_label.tolist())
            # pred
            with torch.no_grad():
                X_input = client.X.clone().detach()
                outputs = self.global_model(X_input)  # o client.model(X_input)
                preds = (outputs.squeeze() > 0.5).int().cpu().numpy()
                y_pred.extend(preds)

        # Y ahora comparas y_pred vs y_true
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        if y_true.shape != y_pred.shape:
            logger.error(f"Shape mismatch: y_true={y_true.shape}, y_pred={y_pred.shape}")

        # Asegúrate de no reusar 'pred_labels' local con 'y_true' global
        avg_loss = float(np.mean(all_losses))
        # Accuracy global
        avg_accuracy = (y_pred == y_true).astype(float).mean()

        precision = precision_score(y_true, y_pred, average="macro")
        recall = recall_score(y_true, y_pred, average="macro")
        f1 = f1_score(y_true, y_pred, average="macro")
        auc = roc_auc_score(y_true, y_pred)

        logger.info(f"Round {round_num} - Loss: {avg_loss}, Acc: {avg_accuracy}, AUC: {auc}")

        return {
            'round': round_num,
            'loss': avg_loss,
            'accuracy': avg_accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'auc': auc
        }

    def select_clients(self, round_num):
        """
        Selecciona 5 clientes en cada ronda, alternando entre clientes con y sin la propiedad.
        """
        clients_with_property = [c for c in self.clients if c.data['has_property']]
        clients_without_property = [c for c in self.clients if not c.data['has_property']]

        # lóg. ejemplo: en rondas pares -> con propiedad
        if round_num % 2 == 0:
            return clients_with_property, True
        else:
            return clients_without_property, False


def normalize_updates(client_updates):
    """
    Normaliza las actualizaciones de los clientes con RobustScaler, si lo deseas.
    """
    scaler = RobustScaler()
    processed_updates = torch.stack([upd.flatten() for upd in client_updates])  # shape [N, dims]
    scaled = scaler.fit_transform(processed_updates.numpy())
    return torch.tensor(scaled)


def average_client_updates(client_updates):
    """
    Promedia las actualizaciones (en la dimensión 0).
    """
    if len(client_updates) == 0:
        raise ValueError("La lista de client_updates está vacía. No se puede calcular el promedio.")

    # normalizar primero, se puede:
    normalized = normalize_updates(client_updates)
    return normalized.mean(dim=0)

    # O si no
    #stacked = torch.stack(client_updates, dim=0)
    #return stacked.mean(dim=0)


In [7]:


#### OUTPUTS ####

# Rondas a ignorar
ronda_ignorar = int(exper_config["rondas_a_ignorar"])


def calculate_correct_predictions(results):
    correct_predictions = 0
    incorrect_predictions = 0
    dynamic_thresholds = calculate_dynamic_threshold(results)

    for i in range(3, len(results)):
        current_prob = results[i]['property_probability']
        previous_prob = results[i - 1]['property_probability']
        previous_property = results[i - 1]['has_property']
        current_property = results[i]['has_property']
        threshold = dynamic_thresholds[i]

        if not previous_property and not current_property:
            is_correct = abs(current_prob - previous_prob) < threshold * 0.1
        elif previous_property and not current_property:
            is_correct = current_prob < previous_prob
        elif not previous_property and current_property:
            is_correct = current_prob > previous_prob
        elif previous_property and current_property:
            is_correct = abs(current_prob - previous_prob) < threshold * 0.1
        else:
            is_correct = False

        if is_correct:
            correct_predictions += 1
        else:
            incorrect_predictions += 1

    total_predictions = correct_predictions + incorrect_predictions
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0

    logger.info(f"Correct predictions: {correct_predictions}, "
                f"Incorrect predictions: {incorrect_predictions}, Accuracy: {accuracy}")
    return correct_predictions, incorrect_predictions, dynamic_thresholds


def plot_results(results, output_path, show=False):
    """
    Genera gráficos de la evolución de la probabilidad detectada, pérdida y precisión.

    Parameters:
        results (list): Resultados de la simulación.
        output_path (str): Directorio para guardar las imágenes.
        show (bool): Si es True, muestra la gráfica al finalizar la ejecución.
    """
    # Ignoramos rondas

    round_nums = [r['round'] for r in results][ronda_ignorar:]
    probabilities = [r['property_probability'] for r in results][ronda_ignorar:]
    thresholds = [r['threshold_used'] for r in results][ronda_ignorar:]
    losses = [r['loss'] for r in results][ronda_ignorar:]
    accuracies = [r['accuracy'] for r in results][ronda_ignorar:]
    property_present = [r['has_property'] for r in results][ronda_ignorar:]

    fig, axs = plt.subplots(2, 1, figsize=(12, 10))

    # Gráfico 1: Probabilidad por ronda
    axs[0].plot(round_nums, probabilities, marker='o', color='blue', label='Property Probability')

    # Agregar puntos en rojo para las rondas donde la propiedad está presente
    has_property_rounds = np.array(round_nums)[property_present]
    has_property_probs = np.array(probabilities)[property_present]
    axs[0].scatter(has_property_rounds, has_property_probs, color='red', zorder=5, label='Rounds with Property')

    axs[0].set_xlabel('Round')
    axs[0].set_ylabel('Property Probability')
    axs[0].set_title('Property Probability by Round (ignoring first 3 rounds)')
    axs[0].grid(True)
    axs[0].legend()

    # Gráfico 2: Pérdida y Precisión del modelo global
    ax1 = axs[1]
    ax1.plot(round_nums, losses, label='Loss', color='red', marker='x')
    ax1.set_xlabel('Round')
    ax1.set_ylabel('Loss', color='red')
    ax1.tick_params(axis='y', labelcolor='red')
    ax1.grid(True)

    ax2 = ax1.twinx()
    ax2.plot(round_nums, accuracies, label='Accuracy', color='blue', marker='o')
    ax2.set_ylabel('Accuracy', color='blue')

    plt.savefig(f"{output_path}/property_probability_loss.png")


def plot_combined_roc_threshold(fpr, tpr, thresholds, auc_roc, optimal_threshold, output_path):
    """
    Genera una imagen con dos gráficos:
    - Izquierda: Curva ROC.
    - Derecha: TPR/FPR vs Threshold con umbral óptimo.
    """
    fig, axs = plt.subplots(1, 2, figsize=(14, 6))

    # Gráfico 1: Curva ROC
    axs[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {auc_roc:.2f})')
    axs[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess')
    axs[0].set_xlabel('False Positive Rate')
    axs[0].set_ylabel('True Positive Rate')
    axs[0].set_title('ROC Curve')
    axs[0].legend(loc="lower right")
    axs[0].grid(True)

    # Gráfico 2: TPR/FPR vs Threshold
    axs[1].plot(thresholds, tpr, label='True Positive Rate (TPR)', color='green', lw=2)
    axs[1].plot(thresholds, fpr, label='False Positive Rate (FPR)', color='red', lw=2)
    axs[1].axvline(optimal_threshold, color='blue', linestyle='--',
                   label=f'Optimal Threshold ({optimal_threshold:.2f})')
    axs[1].set_xlabel('Threshold')
    axs[1].set_ylabel('Rate')
    axs[1].set_title('TPR and FPR vs. Threshold')
    axs[1].legend(loc="best")
    axs[1].grid(True)

    fig.tight_layout()
    plt.savefig(f"{output_path}/roc_threshold.png")
    plt.close()


def calculate_and_log_metrics(results):
    y_true = [r['has_property'] for r in results][ronda_ignorar:]
    y_prob = [r['property_probability'] for r in results][ronda_ignorar:]
    dynamic_thresholds = [r.get('threshold_used', 0.5) for r in results][ronda_ignorar:]

    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    precision, recall, pr_thresholds = precision_recall_curve(y_true, y_prob)
    auc_roc = auc(fpr, tpr)
    auc_pr = auc(recall, precision)

    optimal_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[optimal_idx] if len(thresholds) > 0 else 0.5

    y_pred_dynamic = [1 if prob > thresh else 0 for prob, thresh in zip(y_prob, dynamic_thresholds)]
    y_pred_optimal = [1 if prob > optimal_threshold else 0 for prob in y_prob]

    cm = confusion_matrix(y_true, y_pred_dynamic)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    # Fix: Add zero_division parameter to precision_score
    precision_dyn = precision_score(y_true, y_pred_dynamic, zero_division=0)
    recall_dyn = recall_score(y_true, y_pred_dynamic)
    f1_dyn = f1_score(y_true, y_pred_dynamic)

    accuracy = accuracy_score(y_true, y_pred_optimal)

    # Fix: Add zero_division parameter to precision_score
    precision_opt = precision_score(y_true, y_pred_optimal, zero_division=0)
    recall_opt = recall_score(y_true, y_pred_optimal)
    f1_optimal = f1_score(y_true, y_pred_optimal)
    f1_ci_lower, f1_ci_upper = calculate_f1_ci(y_true, y_pred_optimal)
    entropy_val = calculate_entropy(y_prob)

    correct_transitions, incorrect_transitions, _ = calculate_correct_predictions(results)
    total_transitions = correct_transitions + incorrect_transitions
    custom_precision = (correct_transitions / total_transitions) * 100 if total_transitions > 0 else 0

    metrics = {
        'ROC AUC': auc_roc,
        'PR AUC': auc_pr,
        'Optimal Threshold': optimal_threshold,
        'F1-Score (Optimal)': f1_optimal,
        'F1-Score CI (Optimal)': f"{f1_ci_lower} - {f1_ci_upper}",
        'Accuracy': accuracy,
        'Precision (Dynamic)': precision_dyn,
        'Recall (Dynamic)': recall_dyn,
        'F1-Score (Dynamic)': f1_dyn,
        'Precision (Optimal)': precision_opt,
        'Recall (Optimal)': recall_opt,
        'Entropy': entropy_val,
        'True Positives': tp,
        'False Positives': fp,
        'True Negatives': tn,
        'False Negatives': fn,
        'Custom Precision': custom_precision
    }

    return metrics, fpr, tpr, thresholds


def save_results_to_csv(results, transitions, metrics):
    """
    Guarda los resultados en tres CSV:
    - Resultados detallados por ronda.
    - Transiciones entre rondas.
    - Resultados finales (métricas globales).
    """
    if results:
        fieldnames = list(results[0].keys())  # Asegurar que las claves coincidan
    else:
        fieldnames = ['Round', 'Prediction', 'Probability', 'Clients with Property',
                      'Clients without Property', 'Has Property', 'Loss', 'Accuracy']

    # Resultados por Ronda
    with open(f"{PREFIJO_SAVE}/round_results.csv", 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for r in results:
            writer.writerow(r)

    # Transiciones
    if transitions:
        fieldnames_transitions = list(transitions[0].keys())
        with open(f"{PREFIJO_SAVE}/transitions.csv", 'w', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames_transitions)
            writer.writeheader()
            for t in transitions:
                writer.writerow(t)

    # Resultados Finales
    with open(f"{PREFIJO_SAVE}/final_metrics.csv", 'w', newline='') as csvfile:
        fieldnames_metrics = list(metrics.keys())
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames_metrics)
        writer.writeheader()
        writer.writerow(metrics)


def calculate_dynamic_threshold(results, min_adjustment=exper_config["prob_range"]):
    """
    Calcula un threshold dinámico basado en la evolución de la probabilidad detectada.

    - Si la predicción es incorrecta, ajusta el threshold para mejorar la precisión.
    - Usa un ajuste mínimo `min_adjustment` para evitar cambios bruscos.

    Parameters:
        results (list): Resultados de la simulación.
        min_adjustment (float): Valor mínimo de ajuste del threshold.

    Returns:
        list: Lista de thresholds dinámicos por ronda.
    """
    thresholds = [exper_config["property_threshold"]]

    for i in range(1, len(results)):
        if not isinstance(i, int):
            raise TypeError(f"Expected integer index, got {type(i)} instead.")

        if not isinstance(results, (list, np.ndarray)):
            raise TypeError(f"Expected list or array for results, got {type(results)} instead.")

        prev_prob = results[i - 1]['property_probability']
        curr_prob = results[i]['property_probability']
        actual_property = results[i]['has_property']
        pred_property = results[i]['prediction']

        # Ajuste basado en errores
        if pred_property != actual_property:
            if pred_property:  # Falso positivo, threshold debe subir
                new_threshold = min(thresholds[-1] + min_adjustment, 1.0)
            else:  # Falso negativo, threshold debe bajar
                new_threshold = max(thresholds[-1] - min_adjustment, 0.0)
        else:
            # Ajustar suavemente basado en la evolución de la probabilidad
            if abs(curr_prob - prev_prob) > min_adjustment:
                new_threshold = (curr_prob + prev_prob) / 2
            else:
                new_threshold = thresholds[-1]

        thresholds.append(new_threshold)

    return thresholds


def plot_noisy_vs_clean_accuracy(noisy_results, clean_results, output_path):
    """
    Compara la precisión de FL con y sin ruido en un solo gráfico.
    """
    rounds = range(len(noisy_results))
    noisy_acc = [r['accuracy'] for r in noisy_results]
    clean_acc = [r['accuracy'] for r in clean_results]

    plt.figure(figsize=(10, 5))
    plt.plot(rounds, noisy_acc, label='Con Ruido', color='red')
    plt.plot(rounds, clean_acc, label='Sin Ruido', color='blue')
    plt.xlabel("Rondas")
    plt.ylabel("Precisión")
    plt.title("Impacto del Ruido en la Precisión de FL")
    plt.legend()
    plt.grid()
    plt.savefig(f"{output_path}/noisy_vs_clean_accuracy.png")
    plt.close()


def analyze_probability_transitions(results):
    """
    Analiza las transiciones de probabilidad entre rondas usando filtrado dinámico.

    Parameters:
        results (list): Resultados de la simulación.

    Returns:
        list: Lista de transiciones clasificadas.
    """
    transitions = []
    probabilities = [r['property_probability'] for r in results][ronda_ignorar:]
    rounds = [r['round'] for r in results][ronda_ignorar:]

    # Filtrar valores extremos
    prob_mean = np.mean(probabilities)
    prob_std = np.std(probabilities)
    lower_bound = prob_mean - 2 * prob_std
    upper_bound = prob_mean + 2 * prob_std
    filtered_probs = [p if lower_bound <= p <= upper_bound else prob_mean for p in probabilities]

    min_prob, max_prob = min(filtered_probs), max(filtered_probs)
    range_value = max_prob - min_prob if max_prob > min_prob else 1
    significant_move = range_value * exper_config["prob_range"]

    for i in range(1, len(filtered_probs)):
        prev_prob = filtered_probs[i - 1]
        curr_prob = filtered_probs[i]
        diff = curr_prob - prev_prob
        transition = "stable"

        if abs(diff) > significant_move:
            transition = "increase" if diff > 0 else "decrease"

        transitions.append({
            'round': rounds[i],
            'prev_probability': prev_prob,
            'current_probability': curr_prob,
            'transition': transition
        })

    logger.info(f"Transitions analyzed: {len(transitions)} processed.")
    return transitions


def calculate_f1_ci(y_true, y_pred, confidence_level=0.95, n_resamples=1000):
    f1_scores = [f1_score(y_true, np.random.permutation(y_pred)) for _ in range(n_resamples)]
    ci_lower, ci_upper = np.percentile(f1_scores, [(1 - confidence_level) * 50, (1 + confidence_level) * 50])
    return ci_lower, ci_upper


def calculate_f1_score(y_true, y_pred):
    return f1_score(y_true, y_pred)


def calculate_entropy(probabilities):
    probabilities = np.clip(probabilities, 1e-10, 1 - 1e-10)
    return entropy(probabilities)



In [ ]:
from client import *
from data import *
from salida import *
from server import *
import random


#### EXPERIMENT ####

def set_seed(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "1"


def validate_configuration():
    """
    Valida las configuraciones de parámetros y asegura que las combinaciones sean coherentes.
    """
    logger.info("Validating configuration...")

    # Informar configuraciones redundantes
    if not exper_config["label_flipping"] and exper_config["flipping_antes"]:
        raise ValueError("FLIPPING_ANTES=True no tiene sentido cuando LABEL_FLIPPING=False.")

    if not exper_config["aplicar_ruido"] and not exper_config["label_flipping"]:
        logger.info(
            "Ni ruido ni label flipping están activados. El experimento no incluye perturbaciones en los datos.")

    # Validar parámetros relacionados con ruido
    if exper_config["aplicar_ruido"]:
        if exper_config["epsilon"] <= 0:
            raise ValueError("EPSILON debe ser mayor que 0.")
        if exper_config["delta"] is not None and not (0 < exper_config["delta"] < 1):
            raise ValueError("DELTA debe estar entre 0 y 1 si se usa ruido gaussiano.")
        if exper_config["sensitivity"] <= 0:
            raise ValueError("SENSITIVITY debe ser mayor que 0.")
        if not all(obj in ["gradients", "data"] for obj in exper_config["ruido_obj"]):
            raise ValueError("RUÍDO_OBJ solo puede contener 'gradients' o 'data'.")

    # Validar parámetros relacionados con flipping
    if exper_config["label_flipping"]:
        if not (0 <= exper_config["prob_flip_0"] <= 1):
            raise ValueError("PROB_FLIP_0 debe estar entre 0 y 1.")
        if not (0 <= exper_config["prob_flip_1"] <= 1):
            raise ValueError("PROB_FLIP_1 debe estar entre 0 y 1.")

    logger.info("Configuration validation completed successfully.")


def main():
    set_seed(42)
    validate_configuration()

    # 1) Cargar data
    df = load_data(exper_config['data_file_path'])
    X, y_label, y_slice = preprocess_data(df, exper_config['fraction'])

    # 2) Dividir en global y shadow
    (X_global, y_label_global, y_slice_global), \
        (X_shadow, y_label_shadow, y_slice_shadow), _ = split_data_for_models(
        X, y_label, y_slice
    )

    # 3) Crear datos de clientes
    client_data = create_client_data(X_global, y_label_global, y_slice_global)

    # 4) Modelo global
    global_model = create_global_model(client_data[0]['X'].shape[1])

    # 5) Inicializar clientes
    clients = [SimulatedFlowerClient(i, data, global_model, exper_config, batch_size=exper_config["batch_size"])
               for i, data in enumerate(client_data)]

    # 6) Shadow training → produce (shadow_models, X_attack, y_attack)
    from models import train_shadow_models
    shadow_models, X_attack, y_attack = train_shadow_models(
        X_shadow=torch.tensor(X_shadow, dtype=torch.float32),
        y_label_shadow=torch.tensor(y_label_shadow, dtype=torch.float32),
        y_slice_shadow=torch.tensor(y_slice_shadow, dtype=torch.float32),
        global_model=global_model
    )

    # 7) Entrenamos Attack Model (ejemplo RandomForest)

    X_attack_np = np.stack(X_attack, axis=0)
    y_attack_np = np.array(y_attack)

    input_dim = X_attack_np.shape[1]
    attack_model = AttackModelNN(input_dim)

    # Entrenamos
    attack_model.train_model(
        X_train=X_attack_np,
        y_train=y_attack_np,
        epochs=exper_config["epochs_ataque"],
        lr=exper_config["lr_ataque"],
        batch_size=exper_config["batch_size_attack_model"]
    )
    logger.info("Attack Model NN entrenado con updates etiquetados en PyTorch.")

    # 8) Creamos el servidor pasando attack_model
    from server import FederatedServer
    federated_server = FederatedServer(global_model, clients, attack_model, exper_config['num_rounds'])

    # 9) Ejecución FL
    results = federated_server.train()

    # 10) Salidas, métricas, etc.
    transitions = analyze_probability_transitions(results)
    metrics, fpr, tpr, thresholds = calculate_and_log_metrics(results)
    save_results_to_csv(results, transitions, metrics)
    plot_results(results, exper_config["prefijo_save"])
    plot_combined_roc_threshold(
        fpr, tpr, thresholds,
        metrics['ROC AUC'], metrics['Optimal Threshold'],
        exper_config["prefijo_save"]
    )

    logger.info("Federated learning simulation completed")


In [12]:


if __name__ == "__main__":
    main()



[Grid Search] Ejecutando experimento con config: grid_num20_glo5_sha5_sha0.01_lr_0.001_bat8


python: can't open file '/home/iagobg/notebooks/pruebas pytorch grid/main.py': [Errno 2] No such file or directory


CalledProcessError: Command '['python', 'main.py']' returned non-zero exit status 2.